# Data Management Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Install the datasets library

In [ ]:
```bash

pip install datasets huggingface_hub

In [ ]:
```

### Step 2: Load a dataset

In [ ]:
```python

from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)

print(dataset["train"][0])

In [ ]:
```

This downloads the IMDB movie review dataset. After the first download, it loads from cache at `~/.cache/huggingface/datasets/`.

### Step 3: Stream large datasets

Some datasets are too large to fit on disk. Streaming loads them row by row without downloading the full thing.

In [ ]:
```python

dataset = load_dataset("wikimedia/wikipedia", "20220301.en", split="train", streaming=True)

for i, example in enumerate(dataset):

    print(example["title"])

    if i >= 4:

        break

In [ ]:
```

Streaming gives you an `IterableDataset`. You process rows as they arrive. Memory usage stays constant regardless of dataset size.

### Step 4: Dataset formats

The `datasets` library uses Apache Arrow under the hood. You can convert to other formats depending on what your pipeline needs.

In [ ]:
```python

dataset = load_dataset("imdb", split="train")

dataset.to_csv("imdb_train.csv")

dataset.to_json("imdb_train.json")

dataset.to_parquet("imdb_train.parquet")

In [ ]:
```

Format comparison:

| Format | Size | Read Speed | Best For |

|--------|------|-----------|----------|

| CSV | Large | Slow | Human readability, spreadsheets |

| JSON | Large | Slow | APIs, nested data |

| Parquet | Small | Fast | Analytics, columnar queries |

| Arrow | Small | Fastest | In-memory processing (what `datasets` uses internally) |

For AI work, Parquet is the best storage format. Arrow is what you work with in memory. CSV and JSON are for interchange.

### Step 5: Data splits

Every ML project needs three splits:

- **Train**: The model learns from this (typically 80%)

- **Validation**: You check progress during training (typically 10%)

- **Test**: Final evaluation after training is done (typically 10%)

Some datasets come pre-split. When they don't, split them yourself:

In [ ]:
```python

dataset = load_dataset("imdb", split="train")

split = dataset.train_test_split(test_size=0.2, seed=42)

train_val = split["train"].train_test_split(test_size=0.125, seed=42)

train_ds = train_val["train"]

val_ds = train_val["test"]

test_ds = split["test"]

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
```

Always set a seed for reproducibility. The same seed produces the same split every time.

### Step 6: Download and cache models

Models are large files. The `huggingface_hub` library handles downloading and caching.

In [ ]:
```python

from huggingface_hub import hf_hub_download, snapshot_download

model_path = hf_hub_download(

    repo_id="sentence-transformers/all-MiniLM-L6-v2",

    filename="config.json"

)

print(f"Cached at: {model_path}")

model_dir = snapshot_download("sentence-transformers/all-MiniLM-L6-v2")

print(f"Full model at: {model_dir}")

In [ ]:
```

Models cache to `~/.cache/huggingface/hub/`. Once downloaded, they load instantly on subsequent runs.

### Step 7: Handle large files

Model weights and large datasets should not go into git. Three options:

**Option A: .gitignore (simplest)**

In [ ]:
```

*.bin

*.safetensors

*.pt

*.onnx

data/*.parquet

data/*.csv

models/

In [ ]:
```

**Option B: Git LFS (track large files in git)**

In [ ]:
```bash

git lfs install

git lfs track "*.bin"

git lfs track "*.safetensors"

git add .gitattributes

In [ ]:
```

Git LFS stores pointers in your repo and the actual files on a separate server. GitHub gives you 1 GB free.

**Option C: DVC (data version control)**

In [ ]:
```bash

pip install dvc

dvc init

dvc add data/training_set.parquet

git add data/training_set.parquet.dvc data/.gitignore

git commit -m "Track training data with DVC"

In [ ]:
```

DVC creates small `.dvc` files that point to your data. The data itself lives in S3, GCS, or another remote storage backend.

| Approach | Complexity | Best For |

|----------|-----------|----------|

| .gitignore | Low | Personal projects, downloaded data you can re-fetch |

| Git LFS | Medium | Teams sharing model weights via git |

| DVC | High | Reproducible experiments, large datasets, teams |

For this course, `.gitignore` is enough. Use DVC when you need to reproduce exact experiments across machines.

### Step 8: Storage patterns

**Local storage** works for datasets under ~10 GB. The HF cache handles this automatically.

**Cloud storage** is for anything larger or shared across machines:

In [ ]:
```python

import os

local_path = os.path.expanduser("~/.cache/huggingface/datasets/")

# s3_path = "s3://my-bucket/datasets/"

# gcs_path = "gs://my-bucket/datasets/"

In [ ]:
```

DVC integrates with S3 and GCS directly:

In [ ]:
```bash

dvc remote add -d myremote s3://my-bucket/dvc-store

dvc push

In [ ]:
```

For this course, local storage is sufficient. Cloud storage becomes relevant when you fine-tune on remote GPU instances.

## Exercises

In [ ]:
1. Load the `glue` dataset with the `mrpc` config and inspect the first 5 examples
2. Stream the `c4` dataset and count how many examples you can process in 10 seconds
3. Convert a dataset to Parquet and compare the file size to CSV
4. Create a 70/15/15 train/val/test split with a fixed seed and verify the sizes